In [1]:
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime
from pathlib import Path
import os
import sys

In [2]:
# Get the directory of the script
script_dir = os.getcwd()

# Get the parent directory of the script
parent_dir = os.path.dirname(script_dir)

# Add the parent directory to sys.path
sys.path.append(parent_dir)

import utils.patch_prep as patch_prep


In [3]:
# -----------------------
# Base directories
# -----------------------
DATA_ROOT = Path("/mnt/d/lding/FA/data/FA_ML_Annabel_20250217/031125data")
RESULTS_ROOT = Path("/mnt/d/lding/FA/analysis_results/FA_ML_Annabel_20250217/031125")
seg_folder_str = "code_org_20250820_seg"
# so mask segmentation will be at cell_mask_folder = str(RESULTS_ROOT / ctrl_y_str / seg_folder_str / "mask") as defined below

# -----------------------
# Run timestamp
# -----------------------
time_str = datetime.now().strftime("%Y%m%d_%H%M")

# -----------------------
# Experiment settings
# -----------------------
patch_size = 32
major_ch = 1

# choose: "ctrl" or "y"
condition = "y"   # was ctr_or_y
mask_ratio = 0.4

start_ind = 0
end_ind = 5


In [4]:

# -----------------------
# Derived names
# -----------------------
ctrl_y_str = f"{condition}_ch{major_ch}_major"

group_prefix = f"{condition}_ch{major_ch}"
group_str = f"{group_prefix}_patches_gridonly_wholecell_pslocation00"

# -----------------------
# Derived patch geometry
# -----------------------
half_ps = patch_size // 2
half_half_ps = patch_size // 4
double_ps = patch_size * 2
double_double_ps = patch_size * 4

# image folder depends on condition
image_folder_map = {
    "ctrl": DATA_ROOT / "Control",
    "y":    DATA_ROOT / "Ycomp",
}
if condition not in image_folder_map:
    raise ValueError(f"condition must be one of {list(image_folder_map)}, got {condition!r}")

image_folder = str(image_folder_map[condition])

# segmentation/masks
cell_mask_folder = str(RESULTS_ROOT / ctrl_y_str / seg_folder_str / "mask")
front_mask_dir = str(DATA_ROOT / "frontmasks")

# outputs
movie_partitioned_data_dir = RESULTS_ROOT / ctrl_y_str / group_str / f"tiff_patches{patch_size}_40p_{time_str}"
movie_plot_dir = RESULTS_ROOT / ctrl_y_str / group_str / f"plot_patches{patch_size}_40p_{time_str}"

movie_partitioned_data_dir.mkdir(parents=True, exist_ok=True)
movie_plot_dir.mkdir(parents=True, exist_ok=True)

movie_partitioned_data_dir = str(movie_partitioned_data_dir)
movie_plot_dir = str(movie_plot_dir)

# -----------------------
# Record table (keep column order stable!)
# -----------------------
record_cols = [
    "image_folder","filename","filenameID","x_c","y_c","rand_angle","rand_tx","rand_ty",
    "x_corner1","x_corner2","x_corner3","x_corner4",
    "y_corner1","y_corner2","y_corner3","y_corner4",
    "movie_partitioned_data_dir","crop_img_filename","movie_plot_dir","plot_filename"
]
rows = []


In [5]:

filenames = patch_prep.list_czi_files(image_folder)

debug_flag = 0
rand_trans_flag = 0
rand_rota_flag  = 0

for filenameID in range(start_ind, min(end_ind, len(filenames))):
    filename = filenames[filenameID]

    train_img, train_seg = patch_prep.load_and_pad(
        image_folder, cell_mask_folder, filename, major_ch,
        pad_size=64
    )

    fig_accu, ax_accu = patch_prep.init_debug_fig(train_img, train_seg)

    x_num, y_num, x_0, y_0 = patch_prep.compute_grid(train_img.shape, patch_size)

    for x_i, y_i, x_c, y_c in patch_prep.iter_grid_centers(x_num, y_num, x_0, y_0, patch_size):
        if debug_flag == 1:
            break


        out = patch_prep.extract_big_patch(train_img, train_seg, x_c, y_c, double_ps)
        if out is None:
            continue
        patch_img, patch_seg, x_left, y_left = out

        # quick reject: too little mask in the big patch
        if patch_seg.mean() < mask_ratio / 64:
            continue

        rand_tx, rand_ty = patch_prep.apply_optional_translation(rand_trans_flag, max_shift_px=0)

        big_crop_img, big_crop_seg, (cx_left_1, cy_up_1), _, _ = patch_prep.first_crop_from_big(
            patch_img, patch_seg, patch_size, double_ps, rand_tx, rand_ty
        )

        if big_crop_seg.mean() < mask_ratio / 32:
            continue

        rot_img, rot_seg, rand_angle = patch_prep.apply_optional_rotation(
            big_crop_img, big_crop_seg, rand_rota_flag, max_angle_deg=0.0
        )

        crop_patch_img, crop_patch_seg, (cx_left_2, cy_up_2) = patch_prep.center_crop(
            rot_img, rot_seg, patch_size, half_ps
        )

        if crop_patch_seg.mean() < mask_ratio:
            continue

        crop_img_filename = f"f{str(filenameID).zfill(4)}x{str(x_c).zfill(4)}y{str(y_c).zfill(4)}ps{patch_size}.tif"
        patch_prep.save_patch(movie_partitioned_data_dir, crop_img_filename, crop_patch_img)

        X_full, Y_full = patch_prep.compute_final_polygon_in_full_image(
            patch_size, rand_angle,
            cx_left_2, cy_up_2,
            x_left, y_left,
            cx_left_1, cy_up_1
        )

        plot_filename = f"plot_grid_t{str(filenameID).zfill(4)}_xc{x_c}_yc{y_c}.png"
        ax_accu[0].plot(X_full, Y_full, color='green')
        ax_accu[1].plot(X_full, Y_full, color='green')

        s = patch_prep.make_record_row(
            image_folder, filename, filenameID, x_c, y_c,
            rand_angle, rand_tx, rand_ty,
            X_full, Y_full,
            movie_partitioned_data_dir, crop_img_filename,
            movie_plot_dir, plot_filename
        )
        
        rows.append(s)
    
    data_prep_record = pd.DataFrame(rows, columns=record_cols)

    data_prep_record.to_csv(os.path.join(movie_plot_dir, f"data_prep_record_{group_prefix}_f_{start_ind}_to_{filenameID}.csv"))
    fig_accu.savefig(os.path.join(movie_plot_dir, f"grid_t{str(filenameID).zfill(4)}.png"))
    plt.close(fig_accu)
